In [6]:
"""
rk3.py - 3rd Order Runge-Kutta Method (RK3)
"""

import numpy as np

def rk3(f_ode, xRange, yInitial, numSteps):
    """
    x, y = rk3(f_ode, xRange, yInitial, numSteps)
    
    3rd order Runge-Kutta method (RK3) for solving ODEs.
    Global accuracy: O(h³)
    """
    x = np.zeros(numSteps + 1)
    y = np.zeros((numSteps + 1, np.size(yInitial)))
    dx = (xRange[1] - xRange[0]) / numSteps
    
    for k in range(0, numSteps + 1):
        if k == 0:
            x[0] = xRange[0]
            y[0, :] = yInitial
        else:
            # First intermediate point (midpoint)
            xa = x[k-1] + dx/2
            ya = y[k-1, :] + (dx/2) * f_ode(x[k-1], y[k-1, :])
            
            # Second intermediate point (endpoint with correction)
            xb = x[k-1] + dx
            yb = y[k-1, :] + dx * (2*f_ode(xa, ya) - f_ode(x[k-1], y[k-1, :]))
            
            # Final step using weighted average
            x[k] = x[k-1] + dx
            y[k, :] = y[k-1, :] + (dx/6) * (
                f_ode(x[k-1], y[k-1, :]) + 
                4*f_ode(xa, ya) + 
                f_ode(xb, yb)
            )
    
    if y.shape[1] == 1:
        y = y.flatten()
    
    return x, y

In [7]:
"""
exercise3.py - Exercise 3: RK3 Method (3rd Order Runge-Kutta)
ODE: dy/dx = -y - 3x, y(0) = 1, from x=0 to x=2
"""

import numpy as np
from rk3 import rk3
from expm_ode import expm_ode, exact_solution
from rk2 import rk2

# Problem parameters
xRange = np.array([0.0, 2.0])
yInit = 1.0

# Step sizes
numSteps_list = [10, 20, 40, 80, 160, 320]
h_list = [0.2, 0.1, 0.05, 0.025, 0.0125, 0.00625]

# RK2 errors from Exercise 2
RK2_ERRORS = [4.2255e-03, 1.0451e-03, 2.6061e-04, 6.5121e-05, 1.6286e-05, 4.0713e-06]

exact_at_2 = exact_solution(2.0)

print("\n" + "="*80)
print("EXERCISE 3: RK3 METHOD (3rd ORDER RUNGE-KUTTA)")
print("ODE: dy/dx = -y - 3x, y(0) = 1")
print(f"Exact solution at x = 2.0: y(2) = {exact_at_2:.10f}")
print("="*80)

# ============================================
# TABLE 1: RK3 Results
# ============================================
print("\n" + "="*80)
print("TABLE 1: RK3 RESULTS")
print("="*80)
print(f"{'k':<3} {'numSteps':<10} {'Step size h':<12} {'Y[-1,0]':<18} {'E[k]':<18} {'E[k]/E[k+1]':<15}")
print("-"*80)

errors = []
y_finals = []

for k, numSteps in enumerate(numSteps_list):
    h = h_list[k]
    x, y = rk3(expm_ode, xRange, yInit, numSteps)
    
    # Get final y value
    if y.ndim == 1:
        y_final = y[-1]
    else:
        y_final = y[-1, 0]
    
    y_finals.append(y_final)
    
    # Calculate error
    error = abs(y_final - exact_at_2)
    errors.append(error)
    
    # Calculate ratio
    if k == 0:
        ratio_str = "---"
    else:
        ratio = errors[k-1] / errors[k]
        ratio_str = f"{ratio:.4f}"
    
    print(f"{k:<3} {numSteps:<10} {h:<12.4f} {y_final:<18.10f} {error:<18.10e} {ratio_str:<15}")

print("-"*80)

# ============================================
# Estimate order of accuracy
# ============================================
print("\n" + "="*80)
print("ESTIMATE ORDER OF ACCURACY")
print("="*80)

ratios = [errors[i-1]/errors[i] for i in range(1, len(errors))]
avg_ratio = np.mean(ratios)

print(f"\nError ratios (E[k]/E[k+1]):")
for i, r in enumerate(ratios, 1):
    print(f"  E[{i-1}]/E[{i}] = {r:.4f}")

print(f"\nAverage ratio: {avg_ratio:.4f}")
print(f"Theoretical ratio for 3rd order method: 8.00 (since 2³ = 8)")
print(f"Estimated order p = log2({avg_ratio:.4f}) = {np.log2(avg_ratio):.4f}")
print("\nConclusion: RK3 is 3rd order accurate (p = 3)")

# ============================================
# TABLE 2: Compare RK2 vs RK3
# ============================================
print("\n" + "="*80)
print("TABLE 2: COMPARISON - RK2 vs RK3")
print("="*80)
print(f"{'k':<3} {'numSteps':<10} {'Step size h':<12} {'RK2 Error':<18} {'RK3 Error':<18}")
print("-"*80)

for i in range(len(numSteps_list)):
    print(f"{i:<3} {numSteps_list[i]:<10} {h_list[i]:<12.4f} {RK2_ERRORS[i]:<18.10e} {errors[i]:<18.10e}")

print("-"*80)

# ============================================
# How many steps does RK2 need to match RK3?
# ============================================
print("\n" + "="*80)
print("HOW MANY STEPS DOES RK2 NEED TO MATCH RK3?")
print("="*80)

# Question A: Match RK3 at numSteps=10
rk3_err_10 = errors[0]
print(f"\nA) RK3 with numSteps=10 has error: {rk3_err_10:.10e}")

for i, rk2_err in enumerate(RK2_ERRORS):
    if rk2_err <= rk3_err_10:
        print(f"   RK2 needs numSteps = {numSteps_list[i]} to achieve this accuracy")
        print(f"   RK2 error at {numSteps_list[i]} steps: {rk2_err:.10e}")
        break

# Question B: Match RK3 at numSteps=320
rk3_err_320 = errors[-1]
print(f"\nB) RK3 with numSteps=320 has error: {rk3_err_320:.10e}")

# Error scaling: RK2 error ∝ 1/N², RK3 error ∝ 1/N³
C_RK2 = RK2_ERRORS[-1] * (numSteps_list[-1]**2)
C_RK3 = rk3_err_320 * (numSteps_list[-1]**3)

# For RK2: error = C_RK2 / N²
# We want: C_RK2 / N² = rk3_err_320
required_steps = int(np.ceil(np.sqrt(C_RK2 / rk3_err_320)))

print(f"\n   Error scaling analysis:")
print(f"     RK2: error × N² = {C_RK2:.4e} (constant)")
print(f"     RK3: error × N³ = {C_RK3:.4e} (constant)")
print(f"\n   Required RK2 steps = sqrt(C_RK2 / RK3_error) = {required_steps}")
print(f"   Ratio: RK2 needs {required_steps/320:.1f}× more steps than RK3!")

print(f"\n   Explanation:")
print(f"     RK2 is 2nd order: error ∝ h² ∝ 1/N²")
print(f"     RK3 is 3rd order: error ∝ h³ ∝ 1/N³")
print(f"     To achieve the same error, RK2 needs N^(3/2) steps while RK3 needs N steps")

# ============================================
# VERIFICATION
# ============================================
print("\n" + "="*80)
print("VERIFICATION: RK2 with estimated steps")
print("="*80)

test_steps = [320, 640, 1280, 2560, 5120, required_steps]

print(f"\n{'numSteps':<12} {'RK2 Error':<20} {'Better than RK3(320)?'}")
print("-"*50)

for steps in test_steps:
    x_e, y_e = rk2(expm_ode, xRange, yInit, steps)
    y_final = y_e[-1] if y_e.ndim == 1 else y_e[-1, 0]
    error = abs(y_final - exact_at_2)
    is_better = error < rk3_err_320
    print(f"{steps:<12} {error:<20.10e} {is_better}")

print("\n" + "="*80)
print("Exercise 3 completed!")
print("="*80)

<class 'NameError'>: name 'exact_solution' is not defined